# 04 — Feature Engineering

This notebook prepares the final model-ready dataset from the merged EDA output.

**Some features were already engineered in earlier notebooks:**

- `b2b_games` — back-to-back game counts, calculated from team schedules (NB02)

- `injured_last_season` — binary prior injury flag, created during EDA (NB03)

- `injury_report_count` — renamed from `games_missed` after discovering it measures 
  injury report frequency, not actual games missed (NB03)

**What this notebook adds for our Feature Engineering:**

- Shift target forward (predict *next* season's injuries from *this* season's data)

- Create interaction features (age × minutes, weight × minutes, B2B × minutes)

- Create lag count feature (injury_report_count_last_season)

- Drop leakage, redundant, and identity columns

- Train/test split 

- Handle missing values (fit on train, apply to test)

### Why shift the target forward?
Each row currently has a player's stats AND injuries from the **same season**. But we can't use 
2015-16 stats to predict 2015-16 injuries. That season already happened. So we attach **next season's** 
injury count to each row: 2015-16 stats → predict 2016-17 injuries.

This means 2018-19 rows get dropped (no 2019-20 data to predict), leaving us with:

| Season (features) | Predicting (target) |
|---|---|
| 2013-14 stats | → 2014-15 injuries |
| 2014-15 stats | → 2015-16 injuries |
| 2015-16 stats | → 2016-17 injuries |
| 2016-17 stats | → 2017-18 injuries |
| 2017-18 stats | → 2018-19 injuries |

**Train:** 2013-14 through 2016-17 features → **Test:** 2017-18 features

**Input:** `data/processed/analysis_merged.csv` (3,006 rows, 92 cols)  

**Output:** Train and test CSVs ready for NB05

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# === Configuration ===
PROCESSED_DIR = '../data/processed'
TRAIN_SEASONS = ['2013-14', '2014-15', '2015-16', '2016-17']
TEST_SEASON = '2017-18'  # Last season with known next-season target

pd.set_option('display.max_columns', 50)
print("Ready for feature engineering")

Ready for feature engineering


**Important note on the seasons.** If we're shifting the target forward, the 2018-19 players have no next season data. So our actual usable data is:

- Train: 2013-14 through 2016-17 (features) → predicting 2014-15 through 2017-18 (targets)

- Test: 2017-18 (features) → predicting 2018-19 (target)

Let's load our merged data from NB04

In [5]:
df = pd.read_csv(f'{PROCESSED_DIR}/analysis_merged.csv')
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} cols")
print(f"Seasons: {sorted(df['season'].unique())}")

Loaded: 3,006 rows, 96 cols
Seasons: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19']


---
## Section 1: Shift Target Forward

Like mentioned above: 

Each row currently has a player's stats AND injuries from the **same season**. But we can't use 
2015-16 stats to predict 2015-16 injuries. That season already happened. So we attach **next season's** 
injury count to each row: 2015-16 stats → predict 2016-17 injuries.


For each player, we grab next season's injury_report_count and attache it to the current row. Players whose last season is 2018-19 (or who left the league) get dropped since there's nothing to predict.

In [6]:
# Sort by player and season
df = df.sort_values(['player_id', 'season']).reset_index(drop=True)

# Shift injury_report_count forward: this season's stats → next season's injuries
df['target_next_season'] = df.groupby('player_id')['injury_report_count'].shift(-1)

# How many rows have a valid target?
valid = df['target_next_season'].notna().sum()
dropped = df['target_next_season'].isna().sum()
print(f"Valid rows (have next-season target): {valid:,}")
print(f"Dropped rows (no next season data):  {dropped:,}")
print(f"\nDropped seasons:")
print(df[df['target_next_season'].isna()]['season'].value_counts())

# Drop rows without a target
df = df.dropna(subset=['target_next_season'])
df['target_next_season'] = df['target_next_season'].astype(int)

print(f"\nFinal: {df.shape[0]:,} rows, {df.shape[1]} cols")
print(f"Seasons remaining: {sorted(df['season'].unique())}")

Valid rows (have next-season target): 2,032
Dropped rows (no next season data):  974

Dropped seasons:
season
2018-19    530
2017-18    128
2016-17     86
2015-16     78
2014-15     78
2013-14     74
Name: count, dtype: int64

Final: 2,032 rows, 97 cols
Seasons remaining: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18']


---
## Section 2: Engineer New Features

Let's create two key feature types we want to explore: 

- Injury Report Count From Last Season 

- Feature interactions with Minutes

In [8]:
# Lag feature: injury count from last season (not just binary)
df['injury_report_count_last_season'] = df.groupby('player_id')['injury_report_count'].shift(1)

# Interaction features
df['age_x_minutes'] = df['age'] * df['min']
df['weight_x_minutes'] = df['player_weight'] * df['min']
df['b2b_x_minutes'] = df['b2b_games'] * df['min']
df['age_x_weight'] = df['age'] * df['player_weight']

print("New features created:")
print(f"  injury_report_count_last_season — {df['injury_report_count_last_season'].notna().sum():,} valid, {df['injury_report_count_last_season'].isna().sum()} missing")
print(f"  age_x_minutes     — range: {df['age_x_minutes'].min():.0f} to {df['age_x_minutes'].max():.0f}")
print(f"  weight_x_minutes  — range: {df['weight_x_minutes'].min():.0f} to {df['weight_x_minutes'].max():.0f}")
print(f"  b2b_x_minutes     — range: {df['b2b_x_minutes'].min():.0f} to {df['b2b_x_minutes'].max():.0f}")
print(f"  age_x_weight      — range: {df['age_x_weight'].min():.0f} to {df['age_x_weight'].max():.0f}")

New features created:
  injury_report_count_last_season — 1,332 valid, 700 missing
  age_x_minutes     — range: 12 to 1484
  weight_x_minutes  — range: 109 to 10115
  b2b_x_minutes     — range: 8 to 806
  age_x_weight      — range: 3150 to 9867


Minutes is the common factor because it represents **exposure** — time on the court is when injuries happen.

---
## Section 3: Select Final Features & Drop Columns

Now for our final feature selection and dropping columns that we aren't using

In [9]:
# Define our 15 features
FEATURES = [
    # Workload (5)
    'min', 'gp', 'dist_miles', 'usg_pct', 'ts_pct',
    # Physical (3)
    'age', 'player_height_inches', 'player_weight',
    # Injury history (2)
    'injured_last_season', 'injury_report_count_last_season',
    # Team context (1)
    'b2b_games',
    # Interactions (4)
    'age_x_minutes', 'weight_x_minutes', 'b2b_x_minutes', 'age_x_weight',
]

TARGET = 'target_next_season'

# Keep only what we need + identifiers for splitting
df_model = df[['player_id', 'season'] + FEATURES + [TARGET]].copy()

print(f"Model dataset: {df_model.shape[0]:,} rows, {df_model.shape[1]} cols")
print(f"\n15 features:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")
print(f"\nTarget: {TARGET}")
print(f"Dropped: {df.shape[1] - df_model.shape[1]} columns")

Model dataset: 2,032 rows, 18 cols

15 features:
   1. min
   2. gp
   3. dist_miles
   4. usg_pct
   5. ts_pct
   6. age
   7. player_height_inches
   8. player_weight
   9. injured_last_season
  10. injury_report_count_last_season
  11. b2b_games
  12. age_x_minutes
  13. weight_x_minutes
  14. b2b_x_minutes
  15. age_x_weight

Target: target_next_season
Dropped: 84 columns


---
## Section 4: Train/Test Split

We create our Train/Test Split with a time-based split: Using past seasons to predict the future. (80/20 split)

- Train: 2013-14, 2014-15, 2015-16, 2016-17 (4 seasons)

- Test: 2017-18 (1 season)

So when the model predicts on the test set, it's using 2017-18 data to predict 2018-19 injuries, and we validate against the actual 2018-19 injury counts that are stored in target_next_season.

In [10]:
# Train: 2013-14 through 2016-17 | Test: 2017-18
train = df_model[df_model['season'] != '2017-18'].copy()
test = df_model[df_model['season'] == '2017-18'].copy()

print(f"Train: {train.shape[0]:,} rows ({sorted(train['season'].unique())})")
print(f"Test:  {test.shape[0]:,} rows ({sorted(test['season'].unique())})")
print(f"\nTarget distribution:")
print(f"  Train mean: {train[TARGET].mean():.2f}")
print(f"  Test mean:  {test[TARGET].mean():.2f}")

Train: 1,620 rows (['2013-14', '2014-15', '2015-16', '2016-17'])
Test:  412 rows (['2017-18'])

Target distribution:
  Train mean: 1.41
  Test mean:  1.19


Our 80/20 split target means are similar, meaning no major distribution shift.

---
## Section 5: Handle Missing Values

Some players are missing data — we need to fill those gaps before modeling. We fit imputation on training data only, then apply to test — prevents data leakage.

In [11]:
# Check missing values
print("Missing values:")
for col in FEATURES:
    missing = df_model[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing}")

# Fill injured_last_season and injury_report_count_last_season
# Missing = first season for that player → no prior injury history → 0
train['injured_last_season'] = train['injured_last_season'].fillna(0)
train['injury_report_count_last_season'] = train['injury_report_count_last_season'].fillna(0)
test['injured_last_season'] = test['injured_last_season'].fillna(0)
test['injury_report_count_last_season'] = test['injury_report_count_last_season'].fillna(0)

# Fill height/weight with training median
height_median = train['player_height_inches'].median()
weight_median = train['player_weight'].median()
for dataset in [train, test]:
    dataset['player_height_inches'] = dataset['player_height_inches'].fillna(height_median)
    dataset['player_weight'] = dataset['player_weight'].fillna(weight_median)

# Remaining missing (dist_miles, avg_speed, b2b interactions)
for col in FEATURES:
    if train[col].isna().sum() > 0:
        col_median = train[col].median()
        train[col] = train[col].fillna(col_median)
        test[col] = test[col].fillna(col_median)

# Verify
print(f"\nAfter imputation:")
print(f"  Train missing: {train[FEATURES].isna().sum().sum()}")
print(f"  Test missing:  {test[FEATURES].isna().sum().sum()}")

Missing values:
  player_height_inches: 1
  player_weight: 1
  injured_last_season: 700
  injury_report_count_last_season: 700
  weight_x_minutes: 1
  age_x_weight: 1

After imputation:
  Train missing: 0
  Test missing:  0


---
## Section 6: Save Model-Ready Datasets

Now with our final feature list and our dropped missing values, we're ready to save our model datasets.

In [12]:
# Drop identifiers — models don't need player_id or season
train_final = train.drop(columns=['player_id', 'season'])
test_final = test.drop(columns=['player_id', 'season'])

# Save
train_final.to_csv(f'{PROCESSED_DIR}/train.csv', index=False)
test_final.to_csv(f'{PROCESSED_DIR}/test.csv', index=False)

print(f"Saved train: {train_final.shape[0]:,} rows, {train_final.shape[1]} cols")
print(f"Saved test:  {test_final.shape[0]:,} rows, {test_final.shape[1]} cols")
print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")
print(f"Target: {TARGET}")

Saved train: 1,620 rows, 16 cols
Saved test:  412 rows, 16 cols

Features (15): ['min', 'gp', 'dist_miles', 'usg_pct', 'ts_pct', 'age', 'player_height_inches', 'player_weight', 'injured_last_season', 'injury_report_count_last_season', 'b2b_games', 'age_x_minutes', 'weight_x_minutes', 'b2b_x_minutes', 'age_x_weight']
Target: target_next_season


---
## Summary & Handoff to NB05

### What we did in this notebook
- **Shifted target forward** — each row's target is *next* season's injury report count, 
  so we're predicting the future from current stats
- **Created 5 new features:** `injury_report_count_last_season`, `age_x_minutes`, 
  `weight_x_minutes`, `b2b_x_minutes`, `age_x_weight`
- **Selected 15 features** from 96 columns — dropped 84 columns (leakage, redundant, identity)
- **Train/test split** by time: 2013–2017 features (train) → 2017-18 features (test)
- **Imputed missing values** using training medians only

### Features (15)
| Category | Features |
|----------|----------|
| Workload (5) | min, gp, dist_miles, usg_pct, ts_pct |
| Physical (3) | age, player_height_inches, player_weight |
| Injury history (2) | injured_last_season, injury_report_count_last_season |
| Team context (1) | b2b_games |
| Interactions (4) | age_x_minutes, weight_x_minutes, b2b_x_minutes, age_x_weight |

### Output
- `data/processed/train.csv` — 1,620 rows, 16 cols
- `data/processed/test.csv` — 412 rows, 16 cols

### Next: NB05 (Supervised Modeling)
Three model families: Linear Regression, Random Forest, XGBoost